# Mosaic Holotomography — Reconstruction

The three scans (left, centre, right) are stitched along the distance axis,
giving a single dataset with `ndist_mos = ndist × 3` virtual distances.
The existing `Rec` solver then reconstructs the full `nzobj × nobj × nobj` volume.

In [ ]:
import os
os.environ['CUDA_HOME'] = '/local/nvidia/hpc_sdk/Linux_x86_64/26.1/cuda/13.1'

import numpy as np
import cupy as cp
import h5py
from scipy.fft import fft2, ifft2
from scipy.ndimage import gaussian_filter
from mpi4py import MPI
from types import SimpleNamespace

from holotomocupy.rec_mpi import Rec
from holotomocupy.utils import *

## Acquisition Parameters

Must match the values used in `gen_data.ipynb`.

In [ ]:
n      = 128
ntheta = 360
ndist  = 4

nobj  = 512
nzobj = 192

energy                  = 17.1
detector_pixelsize      = 1.4760147601476e-6 * 2 * 8
focustodetectordistance = 1.217
z1 = np.array([5.110, 5.464, 6.879, 9.817]) * 1e-3

scan_labels = ['left', 'center', 'right']
ndist_mos   = ndist * len(scan_labels)   # 12 virtual distances
z1_mos      = np.tile(z1, len(scan_labels)).astype('float64')  # [12]

## Load and Stitch the Three Scans

Each scan is treated as an additional group of distances:
- `data` : `[ntheta, ndist_mos, nz, n]`
- `ref`  : `[ndist_mos, nz, n]`
- `pos`  : `[ntheta, ndist_mos, 2]`  (axis shift already encoded in pos[:,k,1])

In [ ]:
data_list, ref_list, pos_list = [], [], []

for label in scan_labels:
    with h5py.File(f'data/{label}.h5', 'r') as f:
        data_list.append(f['data'][:])
        ref_list.append(f['ref'][:])
        pos_list.append(f['pos'][:])
        theta = f['theta'][:]

pos_list[0][...,1]+=1
pos_list[0][...,0]-=2

pos_list[2][...,0]+=2
pos_list[2][...,1]-=1


# Stitch along the distance axis
data_mos = np.concatenate(data_list, axis=1)   # [ntheta, ndist_mos, nz, n]
ref_mos  = np.concatenate(ref_list,  axis=0)   # [ndist_mos, nz, n]
pos_mos  = np.concatenate(pos_list,  axis=1)   # [ntheta, ndist_mos, 2]

print(f'data : {data_mos.shape}  {data_mos.dtype}')
print(f'ref  : {ref_mos.shape}')
print(f'pos  : {pos_mos.shape}')
print(f'z1   : {z1_mos}')

mshow(data_mos[0, 0], True)
mshow(ref_mos[0],     True)

## Initialise Rec

In [ ]:
args = SimpleNamespace()

args.energy                  = energy
args.detector_pixelsize      = detector_pixelsize
args.focustodetectordistance = focustodetectordistance
args.z1                      = z1_mos
args.theta                   = theta
args.ndist                   = ndist_mos
args.ntheta                  = ntheta
args.nz                      = n
args.n                       = n
args.nzobj                   = nzobj
args.nobj                    = nobj
args.obj_dtype               = 'complex64'
args.mask                    = 0.9
args.lam_prbfit              = 2e-3
args.rho                     = [1, 0.05, 0.02]
args.niter                   = 257
args.nchunk                  = 4
args.vis_step                = 16
args.err_step                = 16
args.start_iter              = 0
args.comm                    = MPI.COMM_WORLD

cl = Rec(args)

## Initial Guess

- Object  : zeros
- Probe   : flat (1) for all `ndist_mos` virtual distances
- Positions: loaded from HDF5 (include the per-scan axis shifts)

In [ ]:
cl.vars['obj'][:] = 0
cl.vars['prb'][:] = 1
cl.vars['pos'][:] = cp.array(pos_mos)

cl.data[:] = data_mos
cl.ref[:]  = cp.array(ref_mos)

## Reconstruction

In [ ]:
cl.BH()

## Visualise Result

In [ ]:
obj_rec = cl.vars['obj']
mshow_complex(obj_rec[:, nobj//2], True)   # xz mid-slice
mshow_complex(obj_rec[nzobj//2],   True)   # xy mid-slice
mshow_polar(cl.vars['prb'][0],     True)   # first virtual-distance probe